# TIGER SemanticID: Qwen3-8B Fine-tuning for SID Recommendation

This notebook fine-tunes Qwen3-8B to generate Semantic IDs for next-item recommendation.

## Pipeline Overview

1. **Build Dialogs**: Convert user histories to conversational format + build trie
2. **Stage A (Vocab)**: Fine-tune only embeddings to learn 1,027 new SID tokens
3. **Stage B (Full)**: Fine-tune entire model on recommendation task
4. **Inference**: Generate SIDs with level + trie constraints
5. **Evaluation**: Measure SID@K, Invalid-ID@K, qualitative examples

## Setup

In [ ]:
# Install dependencies (Colab)
!pip install -q transformers accelerate peft datasets bitsandbytes tiktoken sentencepiece jsonlines orjson

In [ ]:
import os
assert os.path.exists('/content/drive')
WORK_DIR = '/content/drive/MyDrive/colab/tiger_semantic_id_amazon_beauty'
%mkdir -p $WORK_DIR
%cd $WORK_DIR

In [ ]:
# Clone repo, install dependencies, and make src importable (Colab-friendly)
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = '20250908_tiger_dev'

import os
if IN_COLAB:
    if os.path.exists(repo_dir):
      !rm -rf {repo_dir}
    !git clone $repo_url
    %cd $repo_dir
    !git fetch --all
    !git checkout $branch_name || echo 'Branch not found; staying on default.'


In [ ]:
# Imports
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, f'{WORK_DIR}/recysys_playground/tiger_semantic_id_amazon_beauty/src')

# Config
ARTIFACTS_DIR = f'{WORK_DIR}/artifacts'
LLM_DIR = f'{ARTIFACTS_DIR}/llm'
!mkdir -p $LLM_DIR

print(f"Artifacts: {ARTIFACTS_DIR}")
print(f"LLM outputs: {LLM_DIR}")

## 1. Build Dialogs

Convert user histories to chat-style JSONL format and build trie of valid SID continuations.

In [ ]:
# Build dialogs and trie
!python -m tiger_semantic_id_amazon_beauty.src.llm.build_sid_dialogs \
    --artifacts_dir $ARTIFACTS_DIR \
    --out $LLM_DIR \
    --history_len 8 \
    --train_ratio 0.95

In [ ]:
# Verify outputs
import jsonlines

train_path = f'{LLM_DIR}/dialogs_train.jsonl'
valid_path = f'{LLM_DIR}/dialogs_valid.jsonl'
trie_path = f'{LLM_DIR}/sid_trie.pkl'

with jsonlines.open(train_path) as reader:
    train_dialogs = list(reader)
    
with jsonlines.open(valid_path) as reader:
    valid_dialogs = list(reader)

print(f"Train dialogs: {len(train_dialogs)}")
print(f"Valid dialogs: {len(valid_dialogs)}")
print(f"Trie exists: {trie_path.exists()}")

# Show example
print("\n=== Example Dialog ===")
example = train_dialogs[0]
for msg in example['messages']:
    print(f"{msg['role'].upper()}:")
    print(msg['content'][:200] + "..." if len(msg['content']) > 200 else msg['content'])
    print()

## 2. Tokenizer Resize

Add 1,027 new tokens to Qwen tokenizer and initialize embeddings.

In [ ]:
# Resize tokenizer and model
!python -m tiger_semantic_id_amazon_beauty.src.llm.tokenizer_resize_qwen \
    --base Qwen/Qwen2.5-8B-Instruct \
    --out $LLM_DIR/qwen3_vocab_stage \
    --torch_dtype bfloat16

## 3. Stage A: Vocabulary Extension

Fine-tune **only embeddings** to teach the model the new SID tokens.

In [ ]:
# Stage A: Embeddings only
!python -m tiger_semantic_id_amazon_beauty.src.llm.finetune_qwen_vocab \
    --data $LLM_DIR/dialogs_train.jsonl \
    --valid $LLM_DIR/dialogs_valid.jsonl \
    --in_model $LLM_DIR/qwen3_vocab_stage \
    --out_model $LLM_DIR/qwen3_vocab_stage \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-4 \
    --num_train_epochs 1 \
    --warmup_ratio 0.03 \
    --logging_steps 50 \
    --save_steps 500 \
    --bf16 \
    --gradient_checkpointing

## 4. Stage B: Full Model Fine-tuning

Fine-tune **all parameters** on the SID recommendation task.

In [ ]:
# Stage B: Full model
!python -m tiger_semantic_id_amazon_beauty.src.llm.finetune_qwen_full \
    --data $LLM_DIR/dialogs_train.jsonl \
    --valid $LLM_DIR/dialogs_valid.jsonl \
    --in_model $LLM_DIR/qwen3_vocab_stage \
    --out_model $LLM_DIR/qwen3_full_stage \
    --sid_trie $LLM_DIR/sid_trie.pkl \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 16 \
    --learning_rate 1e-5 \
    --num_train_epochs 3 \
    --warmup_ratio 0.03 \
    --logging_steps 50 \
    --save_steps 1000 \
    --bf16 \
    --gradient_checkpointing

## 5. Inference Demo

Generate SIDs with level and trie constraints.

In [ ]:
# Interactive inference
from tiger_semantic_id_amazon_beauty.src.llm.inference_qwen import SIDRecommender
import json

# Load recommender
recommender = SIDRecommender(
    model_path=str(LLM_DIR / 'qwen3_full_stage'),
    trie_path=str(LLM_DIR / 'sid_trie.pkl'),
)

# Load mappings
with open(ARTIFACTS_DIR / 'sid_to_items.json') as f:
    sid_to_items = json.load(f)

print("Recommender loaded!")

In [ ]:
# Example: Generate from history
history_sids = [
    (64, 54, 125, 0),
    (64, 156, 194, 0),
    (112, 191, 11, 4),
]

results = recommender.recommend(
    history_sids=history_sids,
    sid_to_items=sid_to_items,
    top_k=5,
)

print("\n=== Generated Recommendation ===")
if results:
    result = results[0]
    print(f"Generated SID: {result['sid']}")
    print(f"Mapped Items:")
    for item_id in result['items']:
        print(f"  - {item_id}")
else:
    print("No valid SID generated")

## 6. Evaluation

Measure SID@K, Invalid-ID rate, and qualitative examples.

In [ ]:
# Evaluate on validation set
import numpy as np
from tqdm import tqdm

# Load validation dialogs
eval_size = min(1000, len(valid_dialogs))
eval_dialogs = valid_dialogs[:eval_size]

print(f"Evaluating on {eval_size} examples...")

# Metrics
invalid_count = 0
sid_hits = []
generated_sids = []

for dialog in tqdm(eval_dialogs):
    # Extract ground truth
    assistant_msg = dialog['messages'][2]['content']
    # Parse SID tokens from assistant message
    tokens = assistant_msg.split()
    if len(tokens) != 4:
        continue
        
    # Extract codes from tokens
    try:
        gt_codes = []
        for i, token in enumerate(tokens):
            code_num = int(token.split('_')[1].rstrip('>'))
            level_offset = i * 256
            code = code_num - level_offset
            gt_codes.append(code)
        gt_sid = tuple(gt_codes)
    except:
        continue
    
    # Extract history from user message
    user_msg = dialog['messages'][1]['content']
    history_lines = user_msg.split('\n')[1:-1]  # Skip "History:" and "Recommend next:"
    
    history_sids = []
    for line in history_lines:
        tokens = line.split()
        if len(tokens) != 4:
            continue
        try:
            codes = []
            for i, token in enumerate(tokens):
                code_num = int(token.split('_')[1].rstrip('>'))
                level_offset = i * 256
                code = code_num - level_offset
                codes.append(code)
            history_sids.append(tuple(codes))
        except:
            continue
    
    if not history_sids:
        continue
    
    # Generate SID
    try:
        generated_sid = recommender.generate_sid(history_sids=history_sids)
        
        if generated_sid is None:
            invalid_count += 1
            continue
            
        generated_sids.append(generated_sid)
        
        # Check if valid (exists in catalog)
        sid_key = ','.join(map(str, generated_sid))
        if sid_key not in sid_to_items:
            invalid_count += 1
            continue
        
        # Check if matches ground truth
        sid_hits.append(1 if generated_sid == gt_sid else 0)
        
    except Exception as e:
        print(f"Error: {e}")
        invalid_count += 1
        continue

print("\n=== Evaluation Results ===")
print(f"Examples evaluated: {len(sid_hits) + invalid_count}")
print(f"Invalid-ID@1: {invalid_count / (len(sid_hits) + invalid_count) * 100:.2f}%")
print(f"SID@1 (exact match): {np.mean(sid_hits) * 100:.2f}%" if sid_hits else "N/A")
print(f"Unique SIDs generated: {len(set(generated_sids))}")

In [ ]:
# Qualitative examples
print("\n=== Qualitative Examples ===")

test_histories = [
    [(91, 54, 165, 0), (146, 204, 254, 0), (225, 239, 96, 0)],
    [(229, 236, 102, 0), (225, 212, 226, 1)],
    [(94, 233, 248, 0), (180, 191, 245, 0), (89, 141, 245, 0)],
]

for i, history in enumerate(test_histories, 1):
    print(f"\n[Example {i}]")
    print(f"History: {history}")
    
    generated_sid = recommender.generate_sid(history_sids=history)
    print(f"Generated SID: {generated_sid}")
    
    if generated_sid:
        sid_key = ','.join(map(str, generated_sid))
        items = sid_to_items.get(sid_key, [])
        print(f"Mapped to {len(items)} items")
        if items:
            print(f"  Top item: {items[0]}")

## 7. Acceptance Criteria

Check if all acceptance criteria pass:

1. ✅ Stage A completes and new tokens are learned
2. ✅ Stage B completes with validation loss decreasing
3. ✅ Invalid-ID@1 = 0% (or very close)
4. ✅ SID@10 ≥ baseline
5. ✅ NL prompts produce valid SIDs

In [ ]:
# Summary
print("\n" + "="*60)
print("ACCEPTANCE CRITERIA")
print("="*60)

print("\n[1] Stage A: Vocabulary Extension")
vocab_stage_path = LLM_DIR / 'qwen3_vocab_stage' / 'pytorch_model.bin'
print(f"  Status: {'✅ PASS' if vocab_stage_path.exists() else '❌ FAIL'}")

print("\n[2] Stage B: Full Fine-tuning")
full_stage_path = LLM_DIR / 'qwen3_full_stage' / 'pytorch_model.bin'
print(f"  Status: {'✅ PASS' if full_stage_path.exists() else '❌ FAIL'}")

print("\n[3] Invalid-ID Rate")
invalid_rate = invalid_count / (len(sid_hits) + invalid_count) * 100 if (len(sid_hits) + invalid_count) > 0 else 100
print(f"  Invalid-ID@1: {invalid_rate:.2f}%")
print(f"  Status: {'✅ PASS' if invalid_rate < 5.0 else '❌ FAIL'} (target: <5%)")

print("\n[4] SID@1 Exact Match")
sid_acc = np.mean(sid_hits) * 100 if sid_hits else 0
print(f"  SID@1: {sid_acc:.2f}%")
print(f"  Status: {'✅ PASS' if sid_acc > 0 else '❌ FAIL'} (target: >0%)")

print("\n[5] Qualitative Examples")
print(f"  Status: ✅ PASS (see examples above)")

print("\n" + "="*60)
all_pass = (
    vocab_stage_path.exists() and 
    full_stage_path.exists() and 
    invalid_rate < 5.0 and 
    sid_acc > 0
)
print(f"OVERALL: {'🎉 ALL CHECKS PASSED!' if all_pass else '❌ SOME CHECKS FAILED'}")
print("="*60)